<a href="https://colab.research.google.com/github/Akomon333/NYC-Summer-Youth-Employment-Program-analysis/blob/main/NYC_employment_program_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**About the data**

This data was provided by Department of Youth and Community Development (DYCD) and can be found here https://data.cityofnewyork.us/dataset/Summer-Youth-Employment-Program-SYEP-for-NYCHA-Res/73rz-5b7x

**My goal**

1.   See dynamics of data throughout the years of 5 districts with the most applied to mean value

2.   See dynamics of data throughout the years of 5 districts with the least applied to mean value

3.   Try to find patterns in this dataset




In [1]:
import pandas as pd
import numpy as np
from google.colab import drive

In [ ]:
dataset_unchanged = pd.read_csv("/content/drive/MyDrive/Datasets/NYC_Youth_Employment_Program.csv")
print(dataset_unchanged.columns)
pd.set_option('display.width', 180)      # wider display
pd.set_option('display.max_columns', 10) # show more columns

In [ ]:
dataset = dataset_unchanged.drop(dataset_unchanged[dataset_unchanged['Council District'] == "Total"].index)
dataset = dataset.drop(dataset[dataset['Applied for the program'] == "0"].index)
dataset = dataset.dropna()
dataset.drop(columns=["Average wage of residents"],inplace=True)
dataset['Applied for the program'] = dataset['Applied for the program'].str.replace(',', '').astype(int)
print(dataset["Council District"].unique())

In [ ]:
district_means = dataset.groupby("Council District")["Applied for the program"].mean()
top_five_dseries = district_means.nlargest(5)
least_five_dseries = district_means.nsmallest(5)
print(top_five_dseries)

print(least_five_dseries)

I got the top five and the least five most applied districts.
Now, I will find out how many applicants they received throughout the years in this dataset.

In [ ]:
top_five = dataset[dataset['Council District'].isin(top_five_dseries.index)]
print(top_five.sort_values(by=['Council District','Year']))


In [ ]:
least_five = dataset[dataset['Council District'].isin(least_five_dseries.index)]
print(least_five.sort_values(by=['Council District','Year']))

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [37]:
scaler = StandardScaler()
data = dataset.copy()
columns_to_normalize = ['Year','Applied for the program', 'Were accepted and enrolled', 'Received a referral for social services through the program',
       'Enrolled in financial counseling services through the program', 'Enrolled in college-readiness courses or participated in college-readiness activities through the program']
for col in columns_to_normalize:
    data[col] = data[col].astype(str).str.replace(',', '', regex=False).astype(float)

data[columns_to_normalize] = scaler.fit_transform(data[columns_to_normalize])


In [ ]:
import hdbscan
data_for_hdbscan = data.copy()
clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5)
labels = clusterer.fit_predict(data_for_hdbscan)
data_for_hdbscan['Cluster'] = labels
cluster_means_hdbscan = data_for_hdbscan.groupby('Cluster')[columns_to_normalize].mean()
cluster_means_hdbscan[columns_to_normalize] = scaler.inverse_transform(cluster_means_hdbscan[columns_to_normalize])
data_for_hdbscan[columns_to_normalize] = scaler.inverse_transform(data_for_hdbscan[columns_to_normalize])
print(cluster_means_hdbscan)


In [ ]:
import matplotlib.pyplot as plt
data_for_hdbscan['Council District'] = pd.to_numeric(data_for_hdbscan['Council District'])
data_for_hdbscan_sorted = data_for_hdbscan.sort_values(by='Council District')

plt.figure(figsize=(15, 8))
scatter = plt.scatter(data_for_hdbscan_sorted['Council District'], data_for_hdbscan_sorted['Applied for the program'], c=data_for_hdbscan_sorted['Cluster'], cmap='viridis', s=50, alpha=0.7)
plt.title('HDBSCAN Clustering Results by Council District')
plt.xlabel('Council District')
plt.ylabel('Applied for the program')
plt.xticks(np.arange(0, data_for_hdbscan_sorted['Council District'].max() + 1, 5), rotation=90) # Set ticks at intervals of 5
plt.colorbar(scatter, label='Cluster')
plt.grid(True)
plt.show()

In [ ]:
data_for_hdbscan_with_enrolledinallparts = data_for_hdbscan.copy()
data_for_hdbscan_with_enrolledinallparts['Enrolled in all parts'] = (data_for_hdbscan['Were accepted and enrolled'] + data_for_hdbscan['Received a referral for social services through the program'] + data_for_hdbscan['Enrolled in financial counseling services through the program'] + data_for_hdbscan['Enrolled in college-readiness courses or participated in college-readiness activities through the program'])
plt.figure(figsize=(15, 8))
scatter = plt.scatter(data_for_hdbscan_with_enrolledinallparts['Council District'], data_for_hdbscan_with_enrolledinallparts['Enrolled in all parts'], c=data_for_hdbscan_with_enrolledinallparts['Cluster'], cmap='viridis', s=50, alpha=0.7)
plt.title('HDBSCAN Clustering Results by Council District')
plt.xlabel('Council District')
plt.ylabel('Applied for the program')
plt.xticks(np.arange(0, data_for_hdbscan_with_enrolledinallparts['Council District'].max() + 1, 5), rotation=90) # Set ticks at intervals of 5
plt.colorbar(scatter, label='Cluster')
plt.grid(True)
plt.show()

In [ ]:
columns_to_convert = ['Were accepted and enrolled', 'Received a referral for social services through the program',
       'Enrolled in financial counseling services through the program', 'Enrolled in college-readiness courses or participated in college-readiness activities through the program']

for col in columns_to_convert:
    dataset[col] = dataset[col].astype(str).str.replace(',', '', regex=False).astype(float)

print(dataset.corr().round(3))

In [ ]:
print(least_five.corr().round(3))

In [ ]:
top_five['Council District'] = top_five['Council District'].astype(int)
print(top_five.corr().round(3))

**Conclusion**

1.   Were accepted and enrolled depends a bit(0.137) on the year
2.   Were accepted and enrolled for the least five depends a bit(0.3) on the year and it is almost same(0.318) for the top 5
3.   Council Districts with higher numbers tend to have less(-0.418) applications same for the top and least 5

